In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from splito._scaffold_split import ScaffoldSplit
from rdkit import Chem
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Descriptors
from rdkit import RDLogger

from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import ParameterGrid, train_test_split, StratifiedKFold, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import (matthews_corrcoef, f1_score, confusion_matrix, 
                             accuracy_score, precision_score, recall_score,
                             balanced_accuracy_score, roc_auc_score, make_scorer)
from sklearn.ensemble import RandomForestClassifier, StackingClassifier, VotingClassifier
from sklearn.linear_model import Lasso, LassoCV, ElasticNet, ElasticNetCV

import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import shap

import random
from deap import base, creator, tools, algorithms

import optuna
from optuna.samplers import TPESampler

import warnings
warnings.filterwarnings('ignore')

In [ ]:
def bioactivity_class(value):
    return 'active' if value <= 7500 else 'inactive'

def filter_features(df, exclude_cols, variance_threshold=0.01, correlation_threshold=0.80):
    
    features = df.drop(columns=exclude_cols, errors='ignore')
    num_features = features.select_dtypes(include=[np.number])
    
    selector = VarianceThreshold(threshold=variance_threshold)
    num_array = selector.fit_transform(num_features)
    selected_cols = num_features.columns[selector.get_support()]
    num_filtered = pd.DataFrame(num_array, index=num_features.index, columns=selected_cols)
    
    corr_matrix = num_filtered.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [col for col in upper.columns if any(upper[col] > correlation_threshold)]
    num_filtered = num_filtered.drop(columns=to_drop)
    
    non_num = features.select_dtypes(exclude=[np.number])
    return pd.concat([non_num, num_filtered], axis=1)

def scale_features(df, scaler=None, fit_scaler=False):
    
    num_features = df.select_dtypes(include=[np.number])
    non_binary_cols = [col for col in num_features.columns if num_features[col].nunique() > 2]
    
    if scaler is None:
        scaler = MinMaxScaler()
        fit_scaler = True
    
    if non_binary_cols:
        data = num_features[non_binary_cols].astype(float)
        scaled = scaler.fit_transform(data) if fit_scaler else scaler.transform(data)
        num_features[non_binary_cols] = pd.DataFrame(scaled, index=data.index, columns=non_binary_cols)
    
    non_num = df.select_dtypes(exclude=[np.number])
    return pd.concat([non_num, num_features], axis=1), scaler

def lgb_f1_score_binary(preds, data):
    labels = data.get_label()
    preds_label = (preds > 0.5).astype(int)
    return 'f1', f1_score(labels, preds_label, average='binary'), True

def lgb_mcc_score_binary(preds, data):
    labels = data.get_label()
    preds_label = (preds > 0.5).astype(int)
    return 'mcc', matthews_corrcoef(labels, preds_label), True

def perform_lgb_cv_binary(dtrain, params, additional_info=None, seed=42):
    params = params.copy()
    params['objective'] = 'binary'
    params['seed'] = seed
    
    cv_result = lgb.cv(
        params=params,
        train_set=dtrain,
        num_boost_round=5000,
        nfold=5,
        stratified=True,
        metrics=['binary_logloss'],
        seed=seed,
        feval=[lgb_f1_score_binary, lgb_mcc_score_binary],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=0, first_metric_only=True),
                   lgb.log_evaluation(0)]
    )
    
    best_iter = len(cv_result['valid binary_logloss-mean'])
    result = {
        **params,
        'best_logloss': cv_result['valid binary_logloss-mean'][best_iter-1],
        'best_f1': cv_result['valid f1-mean'][best_iter-1],
        'best_mcc': cv_result['valid mcc-mean'][best_iter-1],
        'best_iteration': best_iter,
        'params': params
    }
    if additional_info:
        result.update(additional_info)
    return result

In [ ]:
DATA_PATH = '../all_feature_train.csv'

EXCLUDE_COLS = ['molecule_chembl_id', 'smiles', 'standard_value', 'class', 
                'pic50', 'scaffold', 'EF', 'scaffold_count']


df = pd.read_csv(DATA_PATH)

df['class'] = df['standard_value'].apply(bioactivity_class)
print(df['class'].value_counts())

df['scaffold'] = df['smiles'].apply(lambda x: MurckoScaffold.MurckoScaffoldSmiles(x))
total_active = len(df[df['class'].isin(['active', 'potent'])])
p_active_total = total_active / len(df)

def calc_ef(group):
    n_scaffold = len(group)
    n_active = len(group[group['class'].isin(['active', 'potent'])])
    p_active = n_active / n_scaffold if n_scaffold > 0 else 0
    return pd.Series({'EF': p_active / p_active_total if p_active_total > 0 else 0,
                      'scaffold_count': n_scaffold})

ef_df = df.groupby('scaffold').apply(calc_ef).reset_index()
df = df.merge(ef_df, on='scaffold', how='left')

df = df[~((df['scaffold_count'] > 10) & (df['EF'] < 0.5))]
df = df[~((df['scaffold_count'] > 5) & (df['EF'] == 0))]
df = df.reset_index(drop=True)


In [ ]:
# Split data using scaffold-aware splitting
spliter = ScaffoldSplit(
    smiles=df['smiles'].values,
    n_splits=5,
    test_size=0.2,
    random_state=100
)
train_idx, test_idx = next(spliter.split(df['class']))

df.loc[train_idx, 'dataset'] = 'train'
df.loc[test_idx, 'dataset'] = 'test'

print(f"Training samples: {len(train_idx)}, Test samples: {len(test_idx)}")
print("\nClass distribution in TRAIN:")
print(df.loc[train_idx, 'class'].value_counts(normalize=True))
print("\nClass distribution in TEST:")
print(df.loc[test_idx, 'class'].value_counts(normalize=True))

In [ ]:
# Remove low-variance and high-correlation features
features_filtered = filter_features(df, EXCLUDE_COLS + ['dataset', 'scaffold_count'])
print(f"Number of features after filtering: {features_filtered.shape[1]}")

# Combine with identifier columns
df_filtered = pd.concat([df[['molecule_chembl_id', 'smiles', 'class', 'dataset']], 
                         features_filtered], axis=1)

# MinMax scaling
train_data = df_filtered[df_filtered['dataset'] == 'train'].drop(columns=['molecule_chembl_id', 'smiles', 'class', 'dataset'])
scaled_train, scaler = scale_features(train_data, fit_scaler=True)

# Apply scaling to the entire dataset
df_scaled = df_filtered.copy()
df_scaled[train_data.columns] = scaler.transform(df_filtered[train_data.columns])

class
inactive    1136
active       565
Name: count, dtype: int64


In [ ]:
TARGET_COL = 'target'
df_scaled['target'] = df_scaled['class'].map({'inactive': 0, 'active': 1})

FEATURES = [col for col in df_scaled.columns if col not in ['molecule_chembl_id', 'smiles', 'class', 'dataset', 'target']]

# Base parameters for LightGBM
base_params = {
    'objective': 'binary',
    'device': 'cpu',
    'num_threads': 16,
    'force_col_wise': True,
    'verbose': -1,
    'seed': 42,
    'min_gain_to_split': 0,
    'min_child_weight': 1,
    'num_leaves': 10,
}

param_grid = {
    'max_depth': [7, 6],
    'learning_rate': [0.03],
    'bagging_fraction': [0.6, 0.8],
    'bagging_freq': [10],
    'feature_fraction': [0.7, 0.85],
    'feature_fraction_bynode': [0.1],
    'lambda_l1': [0, 1],
    'lambda_l2': [0, 1],
}

params_list = list(ParameterGrid(param_grid))
print(f"Total parameter combinations: {len(params_list)}")

In [ ]:
# Create LightGBM dataset for CV
dtrain = lgb.Dataset(
    df_scaled.loc[df_scaled['dataset'] == 'train', FEATURES].values,
    label=df_scaled.loc[df_scaled['dataset'] == 'train', TARGET_COL].values,
    params={'verbose': -1}
)
dtrain.construct()

# Run CV for all combinations
cv_results = []
for i, params in enumerate(params_list):
    result = perform_lgb_cv_binary(
        dtrain, params,
        additional_info={'num_features': dtrain.num_feature(), 
                         'n_dtrain': dtrain.num_data(),
                         'params_id': i}
    )
    cv_results.append(result)

cv_df = pd.DataFrame(cv_results)
print("Best parameter set based on MCC:")
best_row = cv_df.loc[cv_df['best_mcc'].idxmax()]
print(best_row[['params', 'best_mcc', 'best_iteration']])

In [ ]:
# Train model with optimal parameters for SHAP calculation
best_params = best_row['params']
best_iter = int(best_row['best_iteration'])

model = lgb.train(best_params, dtrain, num_boost_round=best_iter)

# Compute SHAP values on the training set
X_train = df_scaled.loc[df_scaled['dataset'] == 'train', FEATURES].values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train)

# Feature importance based on mean absolute SHAP values
feature_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    'feature': FEATURES,
    'shap_value': feature_importance
}).sort_values('shap_value', ascending=False)

print(f"Number of zero-importance features: {sum(shap_df['shap_value'] == 0)}")

In [ ]:


# ----------------------------------------------------------------------
# 1. List of percentiles to evaluate (discrete search space)
# ----------------------------------------------------------------------
PERCENTILE_OPTIONS = [70, 75, 80, 85, 90, 91, 93, 95, 97]

# ----------------------------------------------------------------------
# 2. Evaluation function (identical to manual loop)
# ----------------------------------------------------------------------
def evaluate_percentile(percentile, df_scaled, shap_df, features_all, target_col, best_params, cv_folds=3):
    """
    Evaluate a given percentile threshold using cross-validation.
    Returns mean MCC across CV folds.
    """
    # Select features above the percentile
    threshold = np.percentile(shap_df['shap_value'].values, percentile)
    selected_feats = shap_df[shap_df['shap_value'] > threshold]['feature'].tolist()
    
    if len(selected_feats) < 5:
        return -1.0  # penalty for too few features
    
    X = df_scaled.loc[df_scaled['dataset'] == 'train', selected_feats].values
    y = df_scaled.loc[df_scaled['dataset'] == 'train', target_col].values
    
    dtrain = lgb.Dataset(X, label=y, params={'verbose': -1})
    
    cv_result = lgb.cv(
        params=best_params,
        train_set=dtrain,
        num_boost_round=5000,          # can be reduced for speed
        nfold=cv_folds,
        stratified=True,
        metrics=['binary_logloss'],
        seed=42,
        feval=[lgb_f1_score_binary, lgb_mcc_score_binary],
        callbacks=[lgb.early_stopping(stopping_rounds=20, verbose=0, first_metric_only=True),
                   lgb.log_evaluation(0)]
    )
    
    best_iter = len(cv_result['valid binary_logloss-mean'])
    mcc_mean = cv_result['valid mcc-mean'][best_iter - 1]
    return mcc_mean

# ----------------------------------------------------------------------
# 3. Setup DEAP genetic algorithm (discrete)
# ----------------------------------------------------------------------
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

toolbox = base.Toolbox()

# Attribute: randomly pick a percentile from the list
toolbox.register("attr_percentile", random.choice, PERCENTILE_OPTIONS)
toolbox.register("individual", tools.initRepeat, creator.Individual, toolbox.attr_percentile, 1)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)

def eval_func(individual):
    percentile = individual[0]
    score = evaluate_percentile(
        percentile, df_scaled, shap_df, FEATURES, TARGET_COL, best_params, cv_folds=3
    )
    return (score,)

toolbox.register("evaluate", eval_func)

# Crossover: swap the value between two parents with probability 0.5
def cxUniform(ind1, ind2):
    if random.random() < 0.5:
        ind1[0], ind2[0] = ind2[0], ind1[0]
    return ind1, ind2

# Mutation: change to a different percentile from the list
def mutPercentile(ind):
    current = ind[0]
    options = [p for p in PERCENTILE_OPTIONS if p != current]
    if options:
        ind[0] = random.choice(options)
    return ind,

toolbox.register("mate", cxUniform)
toolbox.register("mutate", mutPercentile)
toolbox.register("select", tools.selTournament, tournsize=3)


# ----------------------------------------------------------------------
# 4. Run the genetic algorithm
# ----------------------------------------------------------------------
population_size = 20
num_generations = 15

pop = toolbox.population(n=population_size)
hof = tools.HallOfFame(1)   # keep only the best individual

stats = tools.Statistics(lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("std", np.std)
stats.register("min", np.min)
stats.register("max", np.max)

pop, log = algorithms.eaSimple(
    pop, toolbox, cxpb=0.5, mutpb=0.2, ngen=num_generations,
    stats=stats, halloffame=hof, verbose=True
)

# ----------------------------------------------------------------------
# 5. Extract the best percentile and select features
# ----------------------------------------------------------------------
best_percentile = hof[0][0]
print(f"\nBest percentile found by GA: {best_percentile:.2f}")

# Select features using the best percentile (exactly as before)
threshold = np.percentile(shap_df['shap_value'].values, best_percentile)
FINAL_FEATURES = shap_df[shap_df['shap_value'] > threshold]['feature'].tolist()
print(f"Number of selected features: {len(FINAL_FEATURES)}")

# Save the final feature list
with open('light - best_features_GA_discrete.txt', 'w') as f:
    for feat in FINAL_FEATURES:
        f.write(feat + '\n')

In [ ]:


# ----------------------------------------------------------------------
# 1. Prepare final dataset (with features selected from GA)
# ----------------------------------------------------------------------
dtrain_final = lgb.Dataset(
    df_scaled.loc[df_scaled['dataset'] == 'train', FINAL_FEATURES].values,
    label=df_scaled.loc[df_scaled['dataset'] == 'train', TARGET_COL].values,
    params={'verbose': -1}
)
dtrain_final.construct()

# ----------------------------------------------------------------------
# 2. Define Optuna objective function
# ----------------------------------------------------------------------
def objective(trial):
    """
    Optuna objective: evaluate a set of hyperparameters using CV and return mean MCC.
    """
    params = {
        'objective': 'binary',
        'device': 'cpu',
        'num_threads': 16,
        'force_col_wise': True,
        'verbose': -1,
        'seed': 42,
        
        # Hyperparameters to optimize
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 8, 64),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.95),
        'feature_fraction_bynode': trial.suggest_float('feature_fraction_bynode', 0.1, 0.8),
        'lambda_l1': trial.suggest_float('lambda_l1', 0.0, 2.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 0.0, 2.0, log=True),
        'min_child_weight': trial.suggest_float('min_child_weight', 0.1, 5.0, log=True),
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.5),
    }
    
    # Run CV with these parameters
    cv_result = lgb.cv(
        params=params,
        train_set=dtrain_final,
        num_boost_round=5000,
        nfold=5,
        stratified=True,
        metrics=['binary_logloss'],
        seed=42,
        feval=[lgb_f1_score_binary, lgb_mcc_score_binary],
        callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=0, first_metric_only=True),
                   lgb.log_evaluation(0)]
    )
    
    # Return mean MCC from the best iteration
    best_iter = len(cv_result['valid binary_logloss-mean'])
    mcc_mean = cv_result['valid mcc-mean'][best_iter - 1]
    return mcc_mean


# ----------------------------------------------------------------------
# 3. Run Optuna optimization
# ----------------------------------------------------------------------
print("Running Optuna optimization...")
study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=42),
    study_name='lgbm_optimization'
)

study.optimize(objective, n_trials=50, show_progress_bar=True)

# Display best parameters from Optuna
best_params_optuna = study.best_params
best_mcc_optuna = study.best_value
print(f"\nBest MCC from Optuna: {best_mcc_optuna:.4f}")
print("Best parameters from Optuna:")
for key, value in best_params_optuna.items():
    print(f"  {key}: {value}")

In [ ]:
# ----------------------------------------------------------------------
# 1. Prepare the final dataset with the features selected from GA
# ----------------------------------------------------------------------
dtrain_final = lgb.Dataset(
    df_scaled.loc[df_scaled['dataset'] == 'train', FINAL_FEATURES].values,
    label=df_scaled.loc[df_scaled['dataset'] == 'train', TARGET_COL].values,
    params={'verbose': -1}
)
dtrain_final.construct()

# ----------------------------------------------------------------------
# 2. Define the refined parameter grid (from your Optuna results)
# ----------------------------------------------------------------------
param_grid_final = {
    'max_depth': [7, 6],
    'num_leaves': [10],                    # should be <= 2^max_depth
    'min_gain_to_split': [0],
    'min_child_weight': [1],
    'learning_rate': [0.03],
    'bagging_fraction': [0.6, 0.8],
    'bagging_freq': [10],
    'feature_fraction': [0.7],
    'feature_fraction_bynode': [0.1],
    'lambda_l1': [0, 1],
    'lambda_l2': [0, 1],
    # Fixed parameters (not tuned)
    'objective': ['binary'],
    'device': ['cpu'],
    'num_threads': [16],
    'force_col_wise': [True],
    'verbose': [-1],
}

# Generate all combinations
final_params_list = list(ParameterGrid(param_grid_final))
print(f"Total parameter combinations: {len(final_params_list)}")

# ----------------------------------------------------------------------
# 3. Evaluate each combination with cross-validation
# ----------------------------------------------------------------------
final_cv_results = []
for i, params in enumerate(final_params_list):
    result = perform_lgb_cv_binary(
        dtrain_final, 
        params,
        additional_info={
            'num_features': dtrain_final.num_feature(),
            'n_dtrain': dtrain_final.num_data(),
            'params_id': i
        }
    )
    final_cv_results.append(result)
    
    # Optional: print progress
    if (i + 1) % 5 == 0 or i == len(final_params_list) - 1:
        print(f"Processed {i+1}/{len(final_params_list)} combinations")

# ----------------------------------------------------------------------
# 4. Find the best combination
# ----------------------------------------------------------------------
final_cv_df = pd.DataFrame(final_cv_results)
best_final_row = final_cv_df.loc[final_cv_df['best_mcc'].idxmax()]

print("\n" + "="*60)
print("FINAL BEST PARAMETERS (from manual grid search)")
print("="*60)
print(f"Best MCC: {best_final_row['best_mcc']:.4f}")
print(f"Best iteration: {best_final_row['best_iteration']}")
print("Best parameters:")
for key, value in best_final_row['params'].items():
    print(f"  {key}: {value}")

FINAL_PARAMS = best_final_row['params']
FINAL_ITER = int(best_final_row['best_iteration'])

In [ ]:
# ----------------------------------------------------------------------
# 1. Train the final model with the best parameters
# ----------------------------------------------------------------------
model_final = lgb.train(
    FINAL_PARAMS,
    dtrain_final,
    num_boost_round=FINAL_ITER
)

# ----------------------------------------------------------------------
# 2. Predict on the test set
# ----------------------------------------------------------------------
X_test = df_scaled.loc[df_scaled['dataset'] == 'test', FINAL_FEATURES].values
y_test = df_scaled.loc[df_scaled['dataset'] == 'test', TARGET_COL].values

y_pred_proba = model_final.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

# ----------------------------------------------------------------------
# 3. Calculate evaluation metrics
# ----------------------------------------------------------------------
metrics = {
    'Accuracy': accuracy_score(y_test, y_pred),
    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
    'MCC': matthews_corrcoef(y_test, y_pred),
    'F1': f1_score(y_test, y_pred),
    'Precision': precision_score(y_test, y_pred),
    'Recall': recall_score(y_test, y_pred),
}

print("\n" + "="*60)
print("FINAL MODEL PERFORMANCE ON TEST SET")
print("="*60)
for k, v in metrics.items():
    print(f"{k:20}: {v:.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

In [85]:
with open('light - 95-259.txt', 'w', encoding='utf-8') as f:
    for item in FEATURES:
        f.write(str(item) + '\n')


## Train Model for prediction

In [ ]:
# ----------------------------------------------------------------------------
# 1. Load feature list and prepare training data
# ----------------------------------------------------------------------------
with open('light - 95-259.txt', 'r', encoding='utf-8') as f:
    FINAL_FEATURES = [line.strip() for line in f]

# Prepare training data with selected features
df_train = pd.concat([
    all_feature[['molecule_chembl_id', 'smiles', 'dataset', 'class']],
    all_feature[FINAL_FEATURES]
], axis=1)

# ----------------------------------------------------------------------------
# 2. Load and prepare herbal dataset
# ----------------------------------------------------------------------------
df_herbal = pd.read_csv("df_fillter_with light_dataset.csv")
df_herbal = df_herbal.rename(columns={"name": "molecule_chembl_id"})
df_herbal['class'] = 'unkhown'

# ----------------------------------------------------------------------------
# 3. Filter features and align datasets
# ----------------------------------------------------------------------------
EXCLUDE_COLS = [
    'molecule_chembl_id', 'smiles', 'standard_value', 'class',
    'pic50', 'stage', 'scaffold_count', 'scaffold', 'EF', 'dataset'
]

feature_filtered = filter_features(df_train, EXCLUDE_COLS)

df_train_filtered = pd.concat([
    df_train[['molecule_chembl_id', 'smiles', 'class', 'dataset']],
    feature_filtered
], axis=1)

df_herbal_filtered = df_herbal[df_train_filtered.columns.tolist()].dropna(axis=0)

# Combine training and herbal data
df_combined = pd.concat([df_train_filtered, df_herbal_filtered], axis=0)

# ----------------------------------------------------------------------------
# 4. Log transformation for skewed features (non-binary columns)
# ----------------------------------------------------------------------------
train_df = df_train_filtered[FINAL_FEATURES]
screen_df = df_herbal_filtered[FINAL_FEATURES]

# Identify column types
non_binary_cols = [col for col in train_df.columns if train_df[col].nunique() > 2]
neg_cols = [col for col in non_binary_cols if train_df[col].min() < 0]
non_neg_cols = list(set(non_binary_cols) - set(neg_cols))

# Custom transformers
def safe_log1p(X):
    X = X.copy()
    X[X <= 0] = 1e-8
    return np.log1p(X)

def shifted_log(X):
    shift = np.abs(X.min()) + 1e-8
    return np.log1p(X + shift)

# Preprocessing pipeline
preprocessor = ColumnTransformer([
    ('shifted_log', FunctionTransformer(shifted_log), neg_cols),
    ('safe_log', FunctionTransformer(safe_log1p), non_neg_cols),
    ('passthrough', 'passthrough', list(set(train_df.columns) - set(non_binary_cols)))
], remainder='drop')

# Transform data
X_train_transformed = preprocessor.fit_transform(train_df)
X_screen_transformed = preprocessor.transform(screen_df)

# Convert to DataFrames
transformed_cols = neg_cols + non_neg_cols + list(set(train_df.columns) - set(non_binary_cols))
X_train_log = pd.DataFrame(X_train_transformed, columns=transformed_cols, index=train_df.index)
X_screen_log = pd.DataFrame(X_screen_transformed, columns=transformed_cols, index=screen_df.index)

# Optional: visualize transformations
def plot_transformation(original, transformed, col_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.hist(original, bins=50, color='blue', alpha=0.7)
    ax1.set_title(f'Original {col_name}')
    ax2.hist(transformed, bins=50, color='red', alpha=0.7)
    ax2.set_title(f'Transformed {col_name}')
    plt.tight_layout()
    plt.show()

for col in non_binary_cols[:6]:
    plot_transformation(train_df[col].values, X_train_log[col].values, col)

# Combine logs back to main DataFrame
df_combined_log = pd.concat([X_train_log, X_screen_log], axis=0)
df_combined = df_combined.reset_index(drop=True)
df_combined_log = df_combined_log.reset_index(drop=True)

df_final = pd.concat([
    df_combined[['molecule_chembl_id', 'smiles', 'class', 'dataset']],
    df_combined_log
], axis=1)

df_final['target'] = df_final['class'].map({'inactive': 0, 'active': 1, 'unkhown': 'unkhown'})

In [ ]:
# ----------------------------------------------------------------------------
# 1. Hyperparameter tuning (using a refined grid from Optuna)
# ----------------------------------------------------------------------------
TARGET_COL = 'target'

param_grid = {
    'max_depth': [6],
    'num_leaves': [10],
    'min_gain_to_split': [0],
    'min_child_weight': [1],
    'learning_rate': [0.03],
    'bagging_fraction': [0.6],
    'bagging_freq': [10],
    'feature_fraction': [0.7],
    'feature_fraction_bynode': [0.1],
    'lambda_l1': [1],
    'lambda_l2': [1],
    'objective': ['binary'],
    'device': ['cpu'],
    'num_threads': [16],
    'force_col_wise': [True],
    'verbose': [-1],
}

params_list = list(ParameterGrid(param_grid))
print(f"Total parameter combinations: {len(params_list)}")

# Prepare dataset for training
dtrain = lgb.Dataset(
    df_final.loc[df_final['dataset'] == 'train', FINAL_FEATURES].values,
    label=df_final.loc[df_final['dataset'] == 'train', TARGET_COL].values,
    params={'verbose': -1}
)
dtrain.construct()

# CV tuning
cv_results = []
for params_id, train_params in tqdm(enumerate(params_list), total=len(params_list), desc="Tuning"):
    result = perform_lgb_cv_binary(
        dtrain, train_params,
        additional_info={
            'num_features': dtrain.num_feature(),
            'n_dtrain': dtrain.num_data(),
            'params_id': params_id
        }
    )
    cv_results.append(result)

# Find best parameters
cv_df = pd.DataFrame(cv_results)
best_row = cv_df.loc[cv_df['best_mcc'].idxmax()]
print(f"Best MCC: {best_row['best_mcc']:.4f}")
print(f"Best iteration: {best_row['best_iteration']}")
print("Best params:", best_row['params'])

# ----------------------------------------------------------------------------
# 2. Train final model and predict on test set
# ----------------------------------------------------------------------------
model = lgb.train(
    best_row['params'],
    dtrain,
    num_boost_round=int(best_row['best_iteration'])
)

# Predict on test set
test_indices = df_final.loc[df_final['dataset'] == 'test'].index.tolist()
X_test = df_final.loc[test_indices, FINAL_FEATURES].values
y_pred_proba = model.predict(X_test)

# Create submission DataFrame
submit = pd.DataFrame({
    'Index': test_indices,
    'Predictions': y_pred_proba,
    'y_pred': (y_pred_proba >= 0.5).astype(int)
})

# ----------------------------------------------------------------------------
# 3. Predict on herbal compounds (screening set)
# ----------------------------------------------------------------------------
herbal_indices = df_final.loc[df_final['dataset'] == 'herbal'].index.tolist()
X_herbal = df_final.loc[herbal_indices, FINAL_FEATURES].values
y_herbal_proba = model.predict(X_herbal)

submit_herbal = pd.DataFrame({
    'Index': herbal_indices,
    'Predictions': y_herbal_proba,
    'y_pred': (y_herbal_proba >= 0.5).astype(int)
})